# HW6 — 총액 Regression Model 선택 및 API 제공
**이름:** TOSHBOEV UTKIRBEK UKTAM UGLI  
**학번:** 23012866

- 입력 데이터: `sales_clean.csv` (HW5 결과)
- 목표: `quantity`, `unit_price`를 사용하여 `total_sales` 예측
- 재현 조건: 모든 셀은 위에서 아래로 실행하며 **Restart → Run All** 후 저장
- 소스: 교수님 *Note 9-4.pdf*, HW6 단계 ①–⑦

## ① 읽기와 나누기 — 10점
입력 데이터 1,004행에서 `quantity`, `unit_price` 두 열을 입력값 X로 사용하고 `total_sales`를 정답 y로 사용한다. 데이터 20%를 test로 나누며 `random_state=0`으로 고정한다.

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print("data rows:", len(sales))
print("training :", len(X_train), "test :", len(X_test))
assert len(sales) == 1004 and len(X_train) == 803 and len(X_test) == 201

data rows: 1004
training : 803 test : 201


## ② 세 개 모델의 test R² 비교 — 15점
단순 선형회귀, 곱셈 특성(`qp`)을 추가한 선형회귀, 제한 없는 결정트리의 test R²를 같은 test set에서 비교한다.

In [2]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])

lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

linear_r2 = lin.score(X_test, y_test)
engineered_r2 = lin2.score(X2_test, y_test)
tree_r2 = tree.score(X_test, y_test)

print("linear regression test R² :", round(linear_r2, 3))
print("linear regression + qp test R² :", round(engineered_r2, 3))
print("decision tree (no limit) test R² :", round(tree_r2, 3))
print("linear + qp coefficients:", lin2.coef_.round(3),
      "intercept:", round(lin2.intercept_, 1))
assert round(linear_r2, 3) == 0.862
assert round(engineered_r2, 3) == 1.000
assert round(tree_r2, 3) == 0.994

linear regression test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994
linear + qp coefficients: [-0. -0.  1.] intercept: 0.0


## ③ 답 — linear regression이 곱셈을 못 하는 이유 — 10점
일반적인 선형회귀는 `quantity`와 `unit_price`의 가중합만 표현하므로, 두 값의 **곱셈 관계** `total_sales = quantity × unit_price`를 정확히 표현하지 못한다. 따라서 test R²가 **0.862**에 그친다.

`qp = quantity × unit_price`를 입력 특성으로 추가하면 모델이 `total_sales ≈ 1 × qp`를 그대로 학습하므로 test R²가 **1.000**이 된다. 예측용 서버에서도 반드시 동일한 `qp`를 계산해야 한다.

## ④ max_depth별 training·test R² 표 — 10점
깊이 `[1, 2, 3, 5, 8, 12, None]`에 대한 잎 개수와 점수를 비교한다.

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([
        str(dep), m.get_n_leaves(),
        round(m.score(X_train, y_train), 3),
        round(m.score(X_test, y_test), 3),
    ])
depth_table = pd.DataFrame(
    rows, columns=["max_depth", "leaves", "training R²", "test R²"]
)
display(depth_table)
assert depth_table["max_depth"].tolist() == ['1','2','3','5','8','12','None']
assert depth_table["leaves"].tolist() == [2,4,8,32,250,781,800]

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## ⑤ cross-validation 및 max_depth 결정 — 10점
전체 데이터에 대해 5-fold 교차검증을 실행하여 평균 R²와 표준편차를 비교한다. 숫자를 보고 깊이 하나를 선택하며 이유를 적는다.

In [4]:
cv_rows = []
for dep in [3, 5, 8, 12, None]:
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5
    )
    cv_rows.append([str(dep), scores.mean(), scores.std()])
    print(f"max_depth {str(dep):4s} mean {scores.mean():.3f} std {scores.std():.3f}")
cv_table = pd.DataFrame(cv_rows, columns=["max_depth", "CV mean R²", "CV std"])
assert round(cv_rows[2][1], 3) == 0.992
assert round(cv_rows[3][1], 3) == 0.995

max_depth 3    mean 0.845 std 0.019
max_depth 5    mean 0.961 std 0.003
max_depth 8    mean 0.992 std 0.000
max_depth 12   mean 0.995 std 0.000
max_depth None mean 0.994 std 0.001


**결정:** 결정트리 중에는 **max_depth=8**을 선택한다. 깊이 8의 교차검증 평균 R²는 **0.992**, 깊이 12는 **0.995**로 차이가 겨우 **0.003**인데, 잎 수는 **250개 vs 781개**이므로 점수가 거의 같을 때 더 단순한 깊이 8을 선택한다. 깊이 제한 없는 트리도 training 1.000과 test 0.994로 차이가 작은데, 이는 총액에 실제로 일정한 곱셈 규칙이 있기 때문이다.

다만 최종 **API 모델**은 곱셈 관계를 정확히 표현하는 **linear regression + qp**를 선택한다.

## ⑥ 선택한 모델 저장 및 API 준비 — 25점
선택 모델 `linear regression + qp`를 `sales_model.pkl`로 저장한다. 별도 `app.py`는 같은 파일을 읽고 `POST /api/sales/predict`를 제공한다. Swagger UI (`/api/docs`)에서 Execute한 뒤 **Response body가 보이는 실제 화면**을 `docs.png`로 저장한다.

In [5]:
# ②에서 학습한 linear regression + qp 모델을 저장
joblib.dump(lin2, "sales_model.pkl")
reloaded_model = joblib.load("sales_model.pkl")
example = pd.DataFrame({"quantity": [40], "unit_price": [200000.0]})
example["qp"] = example["quantity"] * example["unit_price"]
example_pred = int(round(reloaded_model.predict(example)[0]))
print("saved model: sales_model.pkl")
print("example order: quantity=40, unit_price=200000")
print("saved model prediction:", example_pred)
assert example_pred == 8000000

saved model: sales_model.pkl
example order: quantity=40, unit_price=200000
saved model prediction: 8000000


## ⑦ 어느 모델을 선택했고 왜? — 10점
**최종 선택은 linear regression + qp**이다. 수량 **40**, 단가 **200,000원**을 넣으면 실제 곱은 **8,000,000원**, `linear + qp` 예측도 **8,000,000원**인데, 결정트리(depth=8)는 **약 7,362,301원**을 예측하여 약 **637,699원** 차이가 난다.

이는 `total_sales = quantity × unit_price`라는 정확한 공식이 있는 문제이므로, 그 곱을 특성으로 만드는 모델이 더 정확하고 간결하기 때문이다.

In [6]:
# ⑦: 참고용 수치 비교 — 결과는 README에도 서술한다.
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
simple_example = example[["quantity", "unit_price"]]
formula = int(simple_example.loc[0, "quantity"] * simple_example.loc[0, "unit_price"])
tree8_pred = int(round(tree8.predict(simple_example)[0]))
print("linear + qp:", example_pred)
print("tree (depth 8):", tree8_pred)
print("quantity × unit_price:", formula)
print("tree absolute error:", abs(formula - tree8_pred))
assert formula == 8000000 and tree8_pred == 7362301

linear + qp: 8000000
tree (depth 8): 7362301
quantity × unit_price: 8000000
tree absolute error: 637699
